# 0. Import Libraries

In [1]:
import os, sys
from pathlib import Path

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),
        os.path.abspath("../src"),
        os.path.abspath("src"),
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

SRC_PATH = add_project_path("model_implementation")
add_project_path("cnn")
add_project_path("rnn")
ROOT = Path(SRC_PATH).parent.resolve()
print("ROOT:", ROOT)

ROOT: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic


In [2]:
from pathlib import Path
from datetime import datetime
import gc
import itertools
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
except Exception as exc:
    tf = None
    layers = models = None
    print("TensorFlow belum tersedia:", exc)

try:
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split
except Exception as exc:
    f1_score = None
    train_test_split = None
    print("scikit-learn belum tersedia:", exc)

from cnn.layers import Conv2D, LocallyConnected2D, MaxPooling2D, AveragePooling2D, GlobalMaxPooling2D, GlobalAveragePooling2D, Flatten, Dense, Sequential
from cnn.utility import image_loader, batch_loader, image_paths, feature_extractor
from common.io import save_json, load_json, load_npy, save_npy
from rnn.preprocess import read_captions, prep_data, load_vocab
from rnn.sequences import align_features_to_captions, teacher_pairs
from rnn.keras_models import build_preinject, compile_model
from rnn.train import grid_cfg
from rnn.weights import export_weights
from rnn.decode import decode_batch
from rnn.caption_decoder import build_decoder
from rnn.evaluate import eval_keras as eval_caption_keras, eval_caption_decoder, hist_sum, seq_tokens
from common.metrics import bleu4_score, meteor_safe

# 1. Global Variables

In [3]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
MAX_CAPTION_LENGTH = 34

np.random.seed(SEED)
plt.style.use("seaborn-v0_8")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FEATURE_DIR = DATA_DIR / "features"
VOCAB_DIR = DATA_DIR / "vocab"
MODEL_DIR = ROOT / "models"
CNN_MODEL_DIR = MODEL_DIR / "cnn"
RNN_MODEL_DIR = MODEL_DIR / "rnn"
REPORT_DIR = ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIG_DIR = REPORT_DIR / "figures"

for path in [FEATURE_DIR, VOCAB_DIR, CNN_MODEL_DIR, RNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

if tf is not None:
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print("GPU Detected:", gpu_devices)
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("No GPU found, defaulting to CPU.")

No GPU found, defaulting to CPU.


# 2. Loading Intel Image Classification Data

In [4]:
def load_intel_dataset(root_path, target_size=(150, 150)):
    X, y = [], []
    root = Path(root_path)
    for category, label in CATEGORIES.items():
        cat_path = root / category
        if not cat_path.exists():
            continue
        print(f"Loading {category}...")
        for image_path in image_paths(cat_path):
            try:
                X.append(image_loader(image_path, target_size=target_size))
                y.append(label)
            except Exception as exc:
                print(f"Error loading {image_path}: {exc}")
    return np.asarray(X, dtype="float32"), np.asarray(y, dtype="int32")

def first_existing_path(candidates):
    for path in candidates:
        if Path(path).exists():
            return Path(path)
    return Path(candidates[0])

TRAIN_DIR = first_existing_path([
    RAW_DIR / "intel" / "seg_train",
    RAW_DIR / "intel" / "seg_train" / "seg_train",
    RAW_DIR / "intel_image_classification" / "seg_train",
    RAW_DIR / "intel_image_classification" / "seg_train" / "seg_train",
    ROOT / "seg_train" / "seg_train",
])
TEST_DIR = first_existing_path([
    RAW_DIR / "intel" / "seg_test",
    RAW_DIR / "intel" / "seg_test" / "seg_test",
    RAW_DIR / "intel_image_classification" / "seg_test",
    RAW_DIR / "intel_image_classification" / "seg_test" / "seg_test",
    ROOT / "seg_test" / "seg_test",
])

X_train = X_val = X_test = y_train = y_val = y_test = None

if TRAIN_DIR.exists() and TEST_DIR.exists() and train_test_split is not None:
    print("--- Loading Training Data ---")
    print("TRAIN_DIR:", TRAIN_DIR)
    X_all, y_all = load_intel_dataset(TRAIN_DIR, target_size=IMAGE_SIZE)
    print("--- Loading Test Data ---")
    print("TEST_DIR:", TEST_DIR)
    X_test, y_test = load_intel_dataset(TEST_DIR, target_size=IMAGE_SIZE)
    X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all)
    print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
else:
    print("Dataset Intel belum tersedia atau scikit-learn belum siap. CNN cells akan dilewati.")

--- Loading Training Data ---
TRAIN_DIR: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/data/raw/intel/seg_train
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...
--- Loading Test Data ---
TEST_DIR: /Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/data/raw/intel/seg_test
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...
Train: (11227, 150, 150, 3) Val: (2807, 150, 150, 3) Test: (3000, 150, 150, 3)


# 3. CNN Keras Training

In [ ]:
def prepare_dataset_safe(X, y, batch_size=32, shuffle=False):
    def generator():
        for i in range(len(X)):
            yield X[i], y[i]
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )
    if shuffle:
        dataset = dataset.shuffle(buffer_size=min(500, len(X)))
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

class LocalConv2D(tf.keras.layers.Layer if tf is not None else object):
    def __init__(self, filters, kernel_size, strides=(1, 1), activation=None, **kwargs):
        super().__init__(**kwargs)
        self.filters = int(filters)
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (int(kernel_size), int(kernel_size))
        self.strides = strides if isinstance(strides, tuple) else (int(strides), int(strides))
        self.activation_name = activation
        self.activation = tf.keras.activations.get(activation) if tf is not None else None

    def build(self, input_shape):
        channels = int(input_shape[-1])
        height, width = int(input_shape[1]), int(input_shape[2])
        k_h, k_w = self.kernel_size
        s_h, s_w = self.strides
        self.out_h = (height - k_h) // s_h + 1
        self.out_w = (width - k_w) // s_w + 1
        self.kernel = self.add_weight(name='kernel', shape=(self.out_h * self.out_w, k_h * k_w * channels, self.filters), initializer='glorot_uniform', trainable=True)
        self.bias = self.add_weight(name='bias', shape=(self.out_h * self.out_w, self.filters), initializer='zeros', trainable=True)

    def call(self, inputs):
        k_h, k_w = self.kernel_size
        s_h, s_w = self.strides
        channels = int(inputs.shape[-1])
        patches = tf.image.extract_patches(inputs, sizes=[1, k_h, k_w, 1], strides=[1, s_h, s_w, 1], rates=[1, 1, 1, 1], padding='VALID')
        batch = tf.shape(inputs)[0]
        patches = tf.reshape(patches, (batch, self.out_h * self.out_w, k_h * k_w * channels))
        output = tf.einsum('bpc,pco->bpo', patches, self.kernel) + self.bias
        output = tf.reshape(output, (batch, self.out_h, self.out_w, self.filters))
        if self.activation is not None:
            output = self.activation(output)
        return output

    def get_config(self):
        config = super().get_config()
        config.update({'filters': self.filters, 'kernel_size': self.kernel_size, 'strides': self.strides, 'activation': self.activation_name})
        return config

if tf is not None:
    tf.keras.utils.get_custom_objects()['LocalConv2D'] = LocalConv2D

def build_keras_cnn(config, local=False):
    model = models.Sequential(name=f"{'Local' if local else 'CNN'}_l{config['num_layers']}_f{config['filters']}_k{config['kernel_size']}_{config['pooling_type']}")
    conv_layer = LocalConv2D if local else layers.Conv2D
    model.add(conv_layer(config['filters'], (config['kernel_size'], config['kernel_size']), activation='relu', input_shape=(*IMAGE_SIZE, 3)))
    for _ in range(config['num_layers'] - 1):
        model.add(conv_layer(config['filters'], (config['kernel_size'], config['kernel_size']), activation='relu'))
        if config['pooling_type'] == 'max':
            model.add(layers.MaxPooling2D((2, 2)))
        else:
            model.add(layers.AveragePooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu', name='dense_hidden'))
    model.add(layers.Dense(len(CATEGORIES), activation='softmax', name='class_output'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_results_registry = []
cnn_histories = []
local_cnn_record = None

if tf is not None and X_train is not None:
    train_ds = prepare_dataset_safe(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
    val_ds = prepare_dataset_safe(X_val, y_val, batch_size=BATCH_SIZE)

    keras_variations = {
        'num_layers': [2, 3],
        'filters': [32, 64],
        'kernel_size': [3, 5],
        'pooling_type': ['max', 'avg'],
    }
    keys, values = zip(*keras_variations.items())
    configs = [dict(zip(keys, item)) for item in itertools.product(*values)]

    for idx, cfg in enumerate(configs):
        print(f"\n--- Running CNN Experiment {idx + 1}/16: {cfg} ---")
        tf.keras.backend.clear_session()
        model = build_keras_cnn(cfg, local=False)
        history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        y_pred = np.argmax(model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1) if X_test is not None else []
        macro_f1 = float(f1_score(y_test, y_pred, average='macro')) if f1_score is not None and X_test is not None else None
        model_path = CNN_MODEL_DIR / f"cnn_exp_{idx + 1:02d}.keras"
        model.save(model_path)
        record = {'index': idx + 1, **cfg, 'model_path': str(model_path), 'macro_f1': macro_f1, 'shared_parameters': True}
        cnn_results_registry.append(record)
        cnn_histories.append({'index': idx + 1, 'config': cfg, 'history': hist_sum(history)})
        gc.collect()

    best_cfg = max(cnn_results_registry, key=lambda item: item.get('macro_f1') or 0.0)
    local_cfg = {key: best_cfg[key] for key in ['num_layers', 'filters', 'kernel_size', 'pooling_type']}
    print("\n--- Running non-shared LocalConv2D variant:", local_cfg, "---")
    tf.keras.backend.clear_session()
    local_model = build_keras_cnn(local_cfg, local=True)
    local_history = local_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
    local_pred = np.argmax(local_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1) if X_test is not None else []
    local_f1 = float(f1_score(y_test, local_pred, average='macro')) if f1_score is not None and X_test is not None else None
    local_path = CNN_MODEL_DIR / 'cnn_local_best_config.keras'
    local_model.save(local_path)
    local_cnn_record = {'config': local_cfg, 'model_path': str(local_path), 'macro_f1': local_f1, 'shared_parameters': False}

    save_json(cnn_results_registry, TABLE_DIR / 'cnn_records.json')
    save_json(cnn_histories, TABLE_DIR / 'cnn_histories.json')
    save_json(local_cnn_record, TABLE_DIR / 'cnn_local_record.json')

    cnn_df = pd.DataFrame(cnn_results_registry)
    cnn_df.to_csv(TABLE_DIR / 'cnn_records.csv', index=False)
    if not cnn_df.empty and 'macro_f1' in cnn_df:
        factor_tables = {
            'cnn_by_num_layers.csv': cnn_df.groupby('num_layers', as_index=False)['macro_f1'].mean(),
            'cnn_by_filters.csv': cnn_df.groupby('filters', as_index=False)['macro_f1'].mean(),
            'cnn_by_kernel_size.csv': cnn_df.groupby('kernel_size', as_index=False)['macro_f1'].mean(),
            'cnn_by_pooling_type.csv': cnn_df.groupby('pooling_type', as_index=False)['macro_f1'].mean(),
        }
        for filename, table in factor_tables.items():
            table.to_csv(TABLE_DIR / filename, index=False)
        plt.figure(figsize=(10, 4))
        plt.bar(cnn_df['index'].astype(str), cnn_df['macro_f1'].fillna(0.0))
        plt.xlabel('CNN experiment')
        plt.ylabel('Macro F1')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'cnn_macro_f1.png', dpi=160)
        plt.show()
else:
    print("CNN training dilewati.")

pd.DataFrame(cnn_results_registry)


--- Running CNN Experiment 1/16: {'num_layers': 2, 'filters': 32, 'kernel_size': 3, 'pooling_type': 'max'} ---
Epoch 1/10


/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


    351/Unknown 181s 511ms/step - accuracy: 0.5217 - loss: 2.0075

/Users/rusmn/Kuliah/SEMESTER 6/Machine Learning/Tubes2/ML-Tubes-2_RecursiveLearnaholic/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


351/351 ━━━━━━━━━━━━━━━━━━━━ 195s 553ms/step - accuracy: 0.6128 - loss: 1.1583 - val_accuracy: 0.7022 - val_loss: 0.8124
Epoch 2/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 212s 602ms/step - accuracy: 0.8059 - loss: 0.5482 - val_accuracy: 0.7406 - val_loss: 0.7451
Epoch 3/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 220s 627ms/step - accuracy: 0.9173 - loss: 0.2550 - val_accuracy: 0.7496 - val_loss: 0.8490
Epoch 4/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 207s 587ms/step - accuracy: 0.9630 - loss: 0.1239 - val_accuracy: 0.7613 - val_loss: 0.9558
Epoch 5/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 189s 539ms/step - accuracy: 0.9870 - loss: 0.0569 - val_accuracy: 0.7346 - val_loss: 1.2115
Epoch 6/10
213/351 ━━━━━━━━━━━━━━━━━━━━ 1:13 534ms/step - accuracy: 0.9906 - loss: 0.0370

# 4. CNN Forward Propagation

In [ ]:
def keras_to_scratch(keras_model):
    scratch_layers = []
    for layer in keras_model.layers:
        name = layer.__class__.__name__
        weights = layer.get_weights()
        activation = getattr(getattr(layer, 'activation', None), '__name__', None)
        if name == 'Conv2D':
            conv = Conv2D(filters=layer.filters, kernel_size=layer.kernel_size, strides=layer.strides, padding=0, activation=activation)
            conv.load_keras(weights)
            scratch_layers.append(conv)
        elif name == 'MaxPooling2D':
            scratch_layers.append(MaxPooling2D(pool_size=layer.pool_size, strides=layer.strides))
        elif name == 'AveragePooling2D':
            scratch_layers.append(AveragePooling2D(pool_size=layer.pool_size, strides=layer.strides))
        elif name == 'Flatten':
            scratch_layers.append(Flatten())
        elif name == 'Dense':
            dense = Dense(units=layer.units, activation=activation)
            dense.load_keras(weights)
            scratch_layers.append(dense)
    return Sequential(scratch_layers)

cnn_manual_comparison = []

if cnn_results_registry and X_test is not None and tf is not None:
    best = max(cnn_results_registry, key=lambda item: item.get('macro_f1') or 0.0)
    best_model = tf.keras.models.load_model(best['model_path'])
    sample_x = X_test[: min(64, len(X_test))]
    sample_y = y_test[: len(sample_x)]
    scratch_model = keras_to_scratch(best_model)
    keras_pred = np.argmax(best_model.predict(sample_x, verbose=0), axis=1)
    scratch_pred = np.argmax(scratch_model.predict(sample_x), axis=1)
    cnn_manual_comparison = [
        {'implementation': 'keras_shared', 'macro_f1': float(f1_score(sample_y, keras_pred, average='macro')), 'params': int(best_model.count_params())},
        {'implementation': 'scratch_numpy_shared', 'macro_f1': float(f1_score(sample_y, scratch_pred, average='macro')), 'params': scratch_model.count_params()},
    ]
    if local_cnn_record is not None:
        cnn_manual_comparison.append({'implementation': 'keras_non_shared_local', 'macro_f1': local_cnn_record['macro_f1'], 'params': int(local_model.count_params())})
    save_json(cnn_manual_comparison, TABLE_DIR / 'cnn_manual_comparison.json')
    pd.DataFrame(cnn_manual_comparison).to_csv(TABLE_DIR / 'cnn_manual_comparison.csv', index=False)
else:
    print("CNN scratch comparison dilewati.")

cnn_manual_comparison

: 

: 

# 5. Caption Preprocessing

In [ ]:
def read_flickr8k_caption_pairs(path):
    pairs = []
    with Path(path).open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file):
            line = line.strip()
            if not line:
                continue
            if line_number == 0 and line.lower().startswith('image,'):
                continue
            if ',' in line:
                image_name, caption = line.split(',', 1)
            else:
                image_name, caption = line.split(None, 1)
            image_id = Path(image_name.split('#')[0]).stem
            pairs.append((image_id, caption))
    return pairs

CAPTION_FILE = RAW_DIR / "flickr8k" / "captions.txt"
CAPTION_PATH = VOCAB_DIR / "caption_sequences.npy"
VOCAB_PATH = VOCAB_DIR / "vocab.json"
CAPTION_ID_PATH = VOCAB_DIR / "caption_image_ids.txt"

caption_sequences = None
caption_image_ids = []
word_to_index = index_to_word = None

if CAPTION_FILE.exists():
    caption_pairs = read_flickr8k_caption_pairs(CAPTION_FILE)
    caption_image_ids = [image_id for image_id, _ in caption_pairs]
    captions = [caption for _, caption in caption_pairs]
    caption_sequences, word_to_index, index_to_word = prep_data(captions, max_length=MAX_CAPTION_LENGTH, out_dir=VOCAB_DIR)
    CAPTION_ID_PATH.write_text('\n'.join(caption_image_ids), encoding='utf-8')
    print("caption pairs:", len(caption_pairs))
    print("unique images:", len(set(caption_image_ids)))
    print("sequences:", caption_sequences.shape)
    print("vocab:", len(word_to_index))
elif CAPTION_PATH.exists() and VOCAB_PATH.exists() and CAPTION_ID_PATH.exists():
    caption_sequences = load_npy(CAPTION_PATH).astype('int32')
    caption_image_ids = [line.strip() for line in CAPTION_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    word_to_index, index_to_word = load_vocab(VOCAB_DIR)
    print("Loaded caption artifacts:", caption_sequences.shape, len(caption_image_ids), len(word_to_index))
else:
    print("Caption artifacts belum tersedia.")

: 

: 

# 6. Flickr8k Feature Extraction

In [ ]:
IMAGE_DIR = RAW_DIR / "flickr8k" / "Images"
FEATURE_PATH = FEATURE_DIR / "flickr8k_features.npy"
FEATURE_ID_PATH = FEATURE_DIR / "flickr8k_image_ids.txt"
ENCODER_PATH = CNN_MODEL_DIR / "flickr8k_inceptionv3_encoder.keras"

features = None
feature_image_ids = []

if tf is not None and IMAGE_DIR.exists():
    img_paths = image_paths(IMAGE_DIR)
    feature_image_ids = [path.stem for path in img_paths]
    if ENCODER_PATH.exists():
        encoder = tf.keras.models.load_model(ENCODER_PATH)
    else:
        encoder = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet', input_shape=(299, 299, 3), pooling='avg')
        encoder.trainable = False
        encoder.save(ENCODER_PATH)
    features = feature_extractor(img_paths, encoder, FEATURE_PATH, target_size=encoder.input_shape[1:3], batch_size=BATCH_SIZE, image_id_path=FEATURE_ID_PATH, preprocess_fn=tf.keras.applications.inception_v3.preprocess_input)
    print("images:", len(feature_image_ids))
    print("features:", features.shape)
elif FEATURE_PATH.exists() and FEATURE_ID_PATH.exists():
    features = load_npy(FEATURE_PATH).astype('float32')
    feature_image_ids = [line.strip() for line in FEATURE_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    print("Loaded existing features:", features.shape, len(feature_image_ids))
else:
    print("Feature extraction dilewati.")

: 

: 

# 7. RNN/LSTM Decoder Training

In [ ]:
def split_arrays(features, captions, val_ratio=0.2):
    rng = np.random.default_rng(SEED)
    order = np.arange(features.shape[0])
    rng.shuffle(order)
    val_count = int(round(len(order) * val_ratio))
    val_idx, train_idx = order[:val_count], order[val_count:]
    return (features[train_idx], captions[train_idx]), (features[val_idx], captions[val_idx])

def train_decoder_model(model, train_features, train_captions, val_data=None, batch_size=32, epochs=10):
    train_inputs, train_targets = teacher_pairs(train_captions)
    validation_data = None
    if val_data is not None:
        val_features, val_captions = val_data
        val_inputs, val_targets = teacher_pairs(val_captions)
        validation_data = ([val_features, val_inputs], val_targets)
    return model.fit([train_features, train_inputs], train_targets, validation_data=validation_data, batch_size=batch_size, epochs=epochs, verbose=1)

aligned_features = None
aligned_captions = None
aligned_caption_ids = []
rnn_records = []

if features is not None and caption_sequences is not None and caption_image_ids and feature_image_ids:
    aligned_features, aligned_captions, aligned_caption_ids, missing_caption_ids = align_features_to_captions(
        features, feature_image_ids, caption_sequences, caption_image_ids
    )
    if missing_caption_ids:
        print('caption rows without image feature:', len(missing_caption_ids))
    print('aligned features/captions:', aligned_features.shape, aligned_captions.shape)

if tf is not None and aligned_features is not None and aligned_captions is not None and word_to_index is not None:
    base_config = {
        'vocab_size': len(word_to_index),
        'feature_dim': int(aligned_features.shape[1]),
        'max_length': int(aligned_captions.shape[1]),
        'caption_length': int(aligned_captions.shape[1] - 1),
        'embed_dim': 256,
        'learning_rate': 1e-3,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
    }
    train_data, val_data = split_arrays(aligned_features, aligned_captions)
    train_features, train_captions = train_data

    for cfg in grid_cfg(base_config):
        print("\n--- Training decoder:", cfg['model_name'], "---")
        model = build_preinject(
            vocab_size=cfg['vocab_size'], feature_dim=cfg['feature_dim'], max_length=cfg['caption_length'],
            embed_dim=cfg['embed_dim'], hidden_size=cfg['hidden_size'], recur_layers=cfg['recur_layers'], recur_type=cfg['recur_type']
        )
        model = compile_model(model, learn_rate=cfg['learning_rate'])
        history = train_decoder_model(model, train_features, train_captions, val_data=val_data, batch_size=cfg['batch_size'], epochs=cfg['epochs'])
        model_path = RNN_MODEL_DIR / cfg['model_name']
        weight_path = RNN_MODEL_DIR / f"{Path(cfg['model_name']).stem}.npz"
        history_path = TABLE_DIR / f"{Path(cfg['model_name']).stem}_history.json"
        model.save(model_path)
        export_weights(model, weight_path)
        save_json(hist_sum(history), history_path)
        rnn_records.append({'config': cfg, 'model_path': str(model_path), 'weight_path': str(weight_path), 'history_path': str(history_path)})

    save_json(rnn_records, TABLE_DIR / 'train_records.json')
    (FEATURE_DIR / 'aligned_caption_image_ids.txt').write_text('\n'.join(aligned_caption_ids), encoding='utf-8')
else:
    print("RNN/LSTM training dilewati.")

pd.DataFrame(rnn_records)

: 

: 

# 8. RNN/LSTM Evaluation

In [ ]:
keras_scores = []
manual_scores = []

if not rnn_records and (TABLE_DIR / 'train_records.json').exists():
    rnn_records = load_json(TABLE_DIR / 'train_records.json')
if features is None and FEATURE_PATH.exists() and FEATURE_ID_PATH.exists():
    features = load_npy(FEATURE_PATH).astype('float32')
    feature_image_ids = [line.strip() for line in FEATURE_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
if caption_sequences is None and CAPTION_PATH.exists() and VOCAB_PATH.exists() and CAPTION_ID_PATH.exists():
    caption_sequences = load_npy(CAPTION_PATH).astype('int32')
    caption_image_ids = [line.strip() for line in CAPTION_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    word_to_index, index_to_word = load_vocab(VOCAB_DIR)
if 'aligned_features' not in globals() or aligned_features is None:
    if features is not None and caption_sequences is not None and caption_image_ids and feature_image_ids:
        aligned_features, aligned_captions, aligned_caption_ids, missing_caption_ids = align_features_to_captions(features, feature_image_ids, caption_sequences, caption_image_ids)

if tf is not None and rnn_records and aligned_features is not None and aligned_captions is not None:
    max_len = int(aligned_captions.shape[1] - 1)
    for record in rnn_records:
        cfg = record['config']
        model = tf.keras.models.load_model(record['model_path'])
        score = eval_caption_keras(model, aligned_features, aligned_captions, word_to_index, index_to_word, max_len)
        score.update({'implementation': 'keras', 'recur_type': cfg['recur_type'], 'recur_layers': cfg['recur_layers'], 'hidden_size': cfg['hidden_size'], 'model_path': record['model_path']})
        keras_scores.append(score)

        decoder = build_decoder(cfg, record['weight_path'])
        manual_score = eval_caption_decoder(decoder, aligned_features, aligned_captions, word_to_index, index_to_word, max_len)
        manual_score.update({'implementation': 'scratch_numpy', 'recur_type': cfg['recur_type'], 'recur_layers': cfg['recur_layers'], 'hidden_size': cfg['hidden_size'], 'weight_path': record['weight_path']})
        manual_scores.append(manual_score)

    save_json(keras_scores, TABLE_DIR / 'rnn_scores.json')
    save_json(manual_scores, TABLE_DIR / 'manual_scores.json')
    # Tabel turunan untuk kebutuhan analisis laporan.
    keras_df = pd.DataFrame(keras_scores)
    manual_df = pd.DataFrame(manual_scores)
    if not keras_df.empty:
        keras_df.to_csv(TABLE_DIR / 'rnn_scores.csv', index=False)
        keras_df.groupby(['recur_type', 'recur_layers'], as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'caption_by_recurrent_layers.csv', index=False)
        keras_df.groupby(['recur_type', 'hidden_size'], as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'caption_by_hidden_size.csv', index=False)
        keras_df.groupby('recur_type', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'rnn_vs_lstm.csv', index=False)
        keras_df[keras_df['recur_type'] == 'rnn'].groupby('recur_layers', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'rnn_layer_comparison.csv', index=False)
        keras_df[keras_df['recur_type'] == 'lstm'].groupby('recur_layers', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'lstm_layer_comparison.csv', index=False)
        keras_df[keras_df['recur_type'] == 'rnn'].groupby('hidden_size', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'rnn_hidden_comparison.csv', index=False)
        keras_df[keras_df['recur_type'] == 'lstm'].groupby('hidden_size', as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'lstm_hidden_comparison.csv', index=False)
    if not manual_df.empty:
        manual_df.to_csv(TABLE_DIR / 'manual_scores.csv', index=False)
    if not keras_df.empty and not manual_df.empty:
        impl_df = pd.concat([keras_df, manual_df], ignore_index=True)
        impl_df.groupby(['implementation', 'recur_type'], as_index=False)[['bleu4', 'meteor', 'runtime_seconds']].mean().to_csv(TABLE_DIR / 'keras_vs_scratch.csv', index=False)
    if keras_scores:
        score_df = pd.DataFrame(keras_scores)
        labels = score_df['recur_type'].astype(str) + '-' + score_df['recur_layers'].astype(str) + 'x-' + score_df['hidden_size'].astype(str)
        plt.figure(figsize=(12, 4))
        plt.bar(labels, score_df['bleu4'])
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('BLEU-4')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'rnn_bleu4.png', dpi=160)
        plt.show()
    for record in rnn_records:
        history_path = Path(record.get('history_path', ''))
        if not history_path.exists():
            continue
        history = load_json(history_path)
        name = Path(record.get('model_path', 'model')).stem
        plt.figure(figsize=(6, 4))
        if 'loss' in history:
            plt.plot(history['loss'], label='train loss')
        if 'val_loss' in history:
            plt.plot(history['val_loss'], label='validation loss')
        plt.title(name)
        plt.xlabel('epoch')
        plt.ylabel('loss')
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / f'{name}_loss.png', dpi=160)
        plt.close()
    if keras_scores:
        score_df = pd.DataFrame(keras_scores)
        labels = score_df['recur_type'].astype(str) + '-' + score_df['recur_layers'].astype(str) + 'x-' + score_df['hidden_size'].astype(str)
        plt.figure(figsize=(12, 4))
        plt.bar(labels, score_df['bleu4'])
        plt.xticks(rotation=45, ha='right')
        plt.ylabel('BLEU-4')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'rnn_bleu4.png', dpi=160)
        plt.show()
else:
    print("Evaluasi RNN/LSTM dilewati.")

pd.DataFrame(keras_scores)

: 

: 

# 9. Caption Length Study and Qualitative Analysis

In [ ]:
length_scores = []
qualitative_rows = []

if keras_scores and aligned_features is not None and aligned_captions is not None:
    best = max(keras_scores, key=lambda row: row.get('bleu4', 0.0))
    model = tf.keras.models.load_model(best['model_path'])
    for max_len in [10, 20, int(aligned_captions.shape[1] - 1)]:
        score = eval_caption_keras(model, aligned_features, aligned_captions, word_to_index, index_to_word, max_len)
        score.update({'max_length': max_len, 'model_path': best['model_path']})
        length_scores.append(score)
    save_json(length_scores, TABLE_DIR / 'length_scores.json')
    pd.DataFrame(length_scores).to_csv(TABLE_DIR / 'length_scores.csv', index=False)
    if length_scores:
        length_df = pd.DataFrame(length_scores)
        plt.figure(figsize=(6, 4))
        plt.plot(length_df['max_length'], length_df['bleu4'], marker='o')
        plt.xlabel('Max caption length')
        plt.ylabel('BLEU-4')
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'length_bleu4.png', dpi=160)
        plt.show()

    best_by_kind = {}
    for score in keras_scores:
        kind = score.get('recur_type')
        if kind and (kind not in best_by_kind or score.get('bleu4', 0.0) > best_by_kind[kind].get('bleu4', 0.0)):
            best_by_kind[kind] = score
    models_by_kind = {kind: tf.keras.models.load_model(item['model_path']) for kind, item in best_by_kind.items()}
    # Pilih contoh kualitas tinggi/sedang/rendah berdasarkan BLEU rata-rata prediksi model terbaik per kind.
    candidate_count = min(100, len(aligned_features))
    candidate_indices = list(range(candidate_count))
    candidate_predictions = {
        kind: decode_batch(model, aligned_features[candidate_indices], word_to_index, index_to_word, int(aligned_captions.shape[1] - 1))
        for kind, model in models_by_kind.items()
    }
    scored_candidates = []
    for pos, idx in enumerate(candidate_indices):
        refs = [seq_tokens(aligned_captions[idx], index_to_word)]
        scores = [bleu4_score(refs, preds[pos]) for preds in candidate_predictions.values()]
        scored_candidates.append((idx, float(np.mean(scores)) if scores else 0.0))
    scored_candidates = sorted(scored_candidates, key=lambda item: item[1])
    if len(scored_candidates) >= 10:
        low = [idx for idx, _ in scored_candidates[:3]]
        mid_start = max((len(scored_candidates) // 2) - 2, 0)
        mid = [idx for idx, _ in scored_candidates[mid_start:mid_start + 4]]
        high = [idx for idx, _ in scored_candidates[-3:]]
        selected = (low + mid + high)[:10]
    else:
        selected = [idx for idx, _ in scored_candidates]
    predictions_by_kind = {
        kind: decode_batch(model, aligned_features[selected], word_to_index, index_to_word, int(aligned_captions.shape[1] - 1))
        for kind, model in models_by_kind.items()
    }
    for pos, idx in enumerate(selected):
        row = {
            'index': int(idx),
            'image_id': aligned_caption_ids[idx] if idx < len(aligned_caption_ids) else '',
            'ground_truth': ' '.join(seq_tokens(aligned_captions[idx], index_to_word)),
        }
        for kind, predictions in predictions_by_kind.items():
            row[f'{kind}_prediction'] = ' '.join(predictions[pos])
        qualitative_rows.append(row)
    save_json(qualitative_rows, TABLE_DIR / 'qualitative_examples.json')
    pd.DataFrame(qualitative_rows).to_csv(TABLE_DIR / 'qualitative_examples.csv', index=False)
else:
    print("Length study dan qualitative analysis dilewati.")

pd.DataFrame(qualitative_rows)

: 

: 

# 10. Summary

In [ ]:
summary = {
    'cnn_experiments': len(cnn_results_registry) if 'cnn_results_registry' in globals() else 0,
    'cnn_scratch_rows': len(cnn_manual_comparison) if 'cnn_manual_comparison' in globals() else 0,
    'features_ready': features is not None if 'features' in globals() else False,
    'captions_ready': caption_sequences is not None if 'caption_sequences' in globals() else False,
    'rnn_records': len(rnn_records) if 'rnn_records' in globals() else 0,
    'keras_scores': len(keras_scores) if 'keras_scores' in globals() else 0,
    'manual_scores': len(manual_scores) if 'manual_scores' in globals() else 0,
    'length_scores': len(length_scores) if 'length_scores' in globals() else 0,
    'qualitative_rows': len(qualitative_rows) if 'qualitative_rows' in globals() else 0,
}
save_json(summary, TABLE_DIR / 'main_summary.json')
summary

: 

: 

# 11. Mandatory Specification Checklist


In [ ]:
mandatory_status = {
    'cnn_16_shared_experiments': len(cnn_results_registry) >= 16 if 'cnn_results_registry' in globals() else False,
    'cnn_keras_vs_scratch': len(cnn_manual_comparison) >= 2 if 'cnn_manual_comparison' in globals() else False,
    'cnn_shared_vs_non_shared': any(row.get('implementation') == 'keras_non_shared_local' for row in cnn_manual_comparison) if 'cnn_manual_comparison' in globals() else False,
    'caption_preprocessing': caption_sequences is not None and word_to_index is not None if 'caption_sequences' in globals() else False,
    'flickr8k_features': features is not None and len(feature_image_ids) > 0 if 'features' in globals() and 'feature_image_ids' in globals() else False,
    'feature_caption_alignment': aligned_features is not None and aligned_captions is not None if 'aligned_features' in globals() else False,
    'six_rnn_six_lstm': len(rnn_records) >= 12 if 'rnn_records' in globals() else False,
    'bleu_meteor_runtime': len(keras_scores) >= 12 and {'bleu4', 'meteor', 'runtime_seconds'}.issubset(set(keras_scores[0].keys())) if 'keras_scores' in globals() and keras_scores else False,
    'keras_vs_scratch_decoder': len(manual_scores) >= 12 if 'manual_scores' in globals() else False,
    'length_study_3_variants': len(length_scores) >= 3 if 'length_scores' in globals() else False,
    'qualitative_10_examples': len(qualitative_rows) >= 10 if 'qualitative_rows' in globals() else False,
}
save_json(mandatory_status, TABLE_DIR / 'mandatory_status.json')
mandatory_status

: 

: 